In [2]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

In [3]:
input_file = Path(r"D:\Proyek FEB\lengkapin data\46%.xlsx")

df = pd.read_excel(input_file)

COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

In [4]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    return text == "" or text.lower() in [
        "nan", "none", "null", "-", "?",
        "??", "???", "n/a", "na",
        "tidak ada", "not found"
    ]


def journal_key_strong(value):
    """
    Key lebih kuat untuk menyamakan jurnal yang terlihat sama,
    tapi beda spasi, newline, simbol, atau tanda kurung singkatan.
    """
    if pd.isna(value):
        return ""
    
    text = str(value)
    
    # Normalisasi unicode
    text = unicodedata.normalize("NFKC", text)
    
    # Hapus karakter tidak terlihat
    text = text.replace("\u00a0", " ")  # non-breaking space
    text = text.replace("\u200b", "")   # zero-width space
    text = text.replace("\ufeff", "")   # BOM
    
    # Lowercase dan rapikan
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    
    # Samakan simbol umum
    text = text.replace("&", "and")
    text = text.replace("–", "-").replace("—", "-")
    
    # Hilangkan singkatan akhir seperti (IJFS), (JPSS), [JPSS]
    text = re.sub(r"\s*\([a-z0-9\-]+\)\s*$", "", text)
    text = re.sub(r"\s*\[[a-z0-9\-]+\]\s*$", "", text)
    
    # Hilangkan bullet/dash di awal
    text = re.sub(r"^[\-\•\●\▪\*]+\s*", "", text)
    
    # Pertahankan huruf, angka, spasi, hyphen, colon
    text = re.sub(r"[^a-z0-9\s\-\:]", "", text)
    
    # Rapikan spasi sekitar tanda baca
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"\s*:\s*", ": ", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [5]:
df_work = df.copy()

df_work["journal_key_strong"] = df_work[COL_JOURNAL].apply(journal_key_strong)
df_work["q_empty"] = df_work[COL_Q].apply(is_empty_value)

In [6]:
def first_non_empty(series):
    values = [
        str(v).strip()
        for v in series
        if not pd.isna(v) and str(v).strip() != ""
    ]
    return values[0] if values else ""


def join_filled_q(series):
    values = sorted(set(
        str(v).strip()
        for v in series
        if not is_empty_value(v)
    ))
    return " | ".join(values)


mixed_journals = (
    df_work.groupby("journal_key_strong", dropna=False)
    .agg(
        rows=("journal_key_strong", "size"),
        empty_q=("q_empty", "sum"),
        filled_q=("q_empty", lambda x: (~x).sum()),
        journal_name=(COL_JOURNAL, first_non_empty),
        q_values=(COL_Q, join_filled_q),
    )
    .reset_index()
)

mixed_journals = mixed_journals[
    (mixed_journals["empty_q"] > 0) &
    (mixed_journals["filled_q"] > 0)
].copy()

mixed_journals.sort_values("empty_q", ascending=False).head(50)

,journal_key_strong,rows,empty_q,filled_q,journal_name,q_values
405,sustainability,33,7,26,Sustainability (Switzerland),Q1
44,benchmarking: an international journal,4,2,2,Benchmarking: An International Journal,Q1


In [7]:
def parse_q_values(q_values):
    if pd.isna(q_values) or str(q_values).strip() == "":
        return []
    
    return [
        q.strip()
        for q in str(q_values).split("|")
        if q.strip()
    ]


mixed_journals["q_list"] = mixed_journals["q_values"].apply(parse_q_values)
mixed_journals["q_count"] = mixed_journals["q_list"].apply(len)

safe_internal = mixed_journals[mixed_journals["q_count"] == 1].copy()
conflict_internal = mixed_journals[mixed_journals["q_count"] > 1].copy()

print("Jurnal aman internal fill:", len(safe_internal))
print("Jurnal konflik:", len(conflict_internal))

Jurnal aman internal fill: 2
Jurnal konflik: 0


In [8]:
safe_internal["fill_q"] = safe_internal["q_list"].apply(lambda x: x[0])

safe_internal_map = (
    safe_internal
    .set_index("journal_key_strong")["fill_q"]
    .to_dict()
)

df_filled = df_work.copy()

internal_fill_count = 0

for idx, row in df_filled.iterrows():
    key = row["journal_key_strong"]
    
    if key not in safe_internal_map:
        continue
    
    if is_empty_value(row[COL_Q]):
        df_filled.at[idx, COL_Q] = safe_internal_map[key]
        internal_fill_count += 1

print("Jumlah Q terisi dari internal fill strong key:", internal_fill_count)

Jumlah Q terisi dari internal fill strong key: 9


In [9]:
manual_internal_override = {
    "buletin ekonomi moneter dan perbankan": "Q3",
    "buku": "?",
}

manual_count = 0

for idx, row in df_filled.iterrows():
    key = row["journal_key_strong"]
    
    if key in manual_internal_override:
        new_q = manual_internal_override[key]
        
        if df_filled.at[idx, COL_Q] != new_q:
            df_filled.at[idx, COL_Q] = new_q
            manual_count += 1

print("Jumlah baris diseragamkan dari manual override:", manual_count)

Jumlah baris diseragamkan dari manual override: 0


In [10]:
df_filled["q_empty"] = df_filled[COL_Q].apply(is_empty_value)

mixed_after = (
    df_filled.groupby("journal_key_strong", dropna=False)
    .agg(
        rows=("journal_key_strong", "size"),
        empty_q=("q_empty", "sum"),
        filled_q=("q_empty", lambda x: (~x).sum()),
        journal_name=(COL_JOURNAL, first_non_empty),
        q_values=(COL_Q, join_filled_q),
    )
    .reset_index()
)

mixed_after = mixed_after[
    (mixed_after["empty_q"] > 0) &
    (mixed_after["filled_q"] > 0)
].copy()

print("Sisa jurnal yang masih campur kosong dan terisi:", len(mixed_after))

mixed_after.sort_values("empty_q", ascending=False).head(50)

Sisa jurnal yang masih campur kosong dan terisi: 0


,journal_key_strong,rows,empty_q,filled_q,journal_name,q_values


In [13]:
df_filled.replace('?', pd.NA, inplace=True)
df_filled.isnull().sum()

No                         0
Title                      0
Authors                    0
Journal                    9
SCOPUS Q                  99
Tahun                      2
Volume / Issue            96
LINK DOI/ARTIKEL          54
SCOPUS SITASI           1524
raw_key                    9
q_source_checkpoint2       0
journal_key_strong         0
q_empty                    0
dtype: int64

In [14]:
df_filled.to_excel("pakai ini yang terbaru.xlsx", index=False)